# Dataset 2 — Embeddings v1 (link-prediction objective)

Mirror of Dataset 1's `03_embeddings_v1`. Builds, at dims 32 / 64 / 128:
- **GraphSAGE v1** (link-prediction) -> `feature_based/`
- **Node2Vec v1** (random walks, structure only) -> `network_based/`

Each candidate is merged with the Dataset 2 baseline target and saved for the p=0 model selection.

In [1]:
import sys
from pathlib import Path

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'requirements.txt').exists():
            return path
    raise FileNotFoundError('Project root not found.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.models.embeddings import GNNConfig, Node2VecConfig, extract_embeddings

DATASET2_PATH = PROJECT_ROOT / 'src' / 'datasets' / 'dataset_2'
EMB_ROOT      = PROJECT_ROOT / 'src' / 'data' / 'embeddings'

def emb_path(name):
    name = str(name)
    ds  = 'dataset_2' if 'dataset2' in name else 'dataset_1'
    sub = 'network_based' if name.startswith('node2vec') else 'feature_based'
    p = EMB_ROOT / ds / sub / name
    p.parent.mkdir(parents=True, exist_ok=True)
    return p

pd.set_option('display.max_columns', 200)
print('Project root:', PROJECT_ROOT)

Project root: /Users/rubenmarques/Documents/Repositórios/Thesis


## Load Dataset 2 graph

In [2]:
def load_dataset2_nodes(dataset_path):
    nodes = pd.read_csv(dataset_path / 'nodes.csv').reset_index(drop=True)
    nodes['index']  = nodes.index
    nodes['Equity'] = nodes['buffer']
    nodes['Assets'] = nodes['assets']
    return nodes

def load_dataset2_edges(dataset_path, bank_to_idx):
    matrix = pd.read_excel(dataset_path / 'network.xlsx', index_col=0)
    edges_long = matrix.stack().reset_index()
    edges_long.columns = ['source_bank', 'target_bank', 'Weights']
    edges_long = edges_long[edges_long['Weights'] != 0].copy()
    edges_long['Sourceid'] = edges_long['source_bank'].map(bank_to_idx)
    edges_long['Targetid'] = edges_long['target_bank'].map(bank_to_idx)
    edges_long = edges_long.dropna(subset=['Sourceid', 'Targetid'])
    edges_long['Sourceid'] = edges_long['Sourceid'].astype(int)
    edges_long['Targetid'] = edges_long['Targetid'].astype(int)
    return edges_long[['Sourceid', 'Targetid', 'Weights']].reset_index(drop=True)

nodes_d2 = load_dataset2_nodes(DATASET2_PATH)
bank_to_idx_d2 = dict(zip(nodes_d2['bank'], nodes_d2['index']))
edges_d2 = load_dataset2_edges(DATASET2_PATH, bank_to_idx_d2)
feature_cols_d2 = ['assets', 'liabilities', 'buffer']  # GraphSAGE node inputs
print(f'Dataset 2: {len(nodes_d2)} banks, {len(edges_d2)} edges')

Dataset 2: 1444 banks, 2893 edges


## Configs (dims 32 / 64 / 128)

In [3]:
DIMS = [32, 64, 128]

def gs_v1(dim):
    return GNNConfig(hidden_dims=(256, dim), dropout=0.3, lr=0.01, epochs=100, aggregation='mean', device='cpu')

def n2v(dim):
    neg = 5 if dim == 128 else 1
    return Node2VecConfig(embedding_dim=dim, walk_length=20, context_size=10, walks_per_node=10,
                          num_negative_samples=neg, batch_size=128, lr=0.01, epochs=100, device='cpu')

## Generate + save

In [4]:
target_baseline = pd.read_csv(DATASET2_PATH / 'targets' / 'target.csv')
target_cols = [c for c in target_baseline.columns if c != 'bank_id']

def build_and_save(emb_df, fname):
    emb_cols = [c for c in emb_df.columns if c.startswith('emb_')]
    merged = emb_df[['bank_id'] + emb_cols].merge(target_baseline, on='bank_id', how='inner')
    merged = merged[['bank_id'] + emb_cols + target_cols]
    out = emb_path(fname)
    merged.to_parquet(out, index=False)
    print(f'{fname:42s}  shape={merged.shape}  -> {out.relative_to(PROJECT_ROOT)}')
    return merged

for dim in DIMS:
    df, _ = extract_embeddings(edges_d2, nodes_d2, gs_v1(dim), feature_cols=feature_cols_d2)
    build_and_save(df, f'graphsage_v1_{dim}_dataset2_dataset.parquet')
    df, _ = extract_embeddings(edges_d2, nodes_d2, n2v(dim))
    build_and_save(df, f'node2vec_v1_{dim}_dataset2_dataset.parquet')

graphsage_v1_32_dataset2_dataset.parquet    shape=(1444, 35)  -> src/data/embeddings/dataset_2/feature_based/graphsage_v1_32_dataset2_dataset.parquet


node2vec_v1_32_dataset2_dataset.parquet     shape=(1444, 35)  -> src/data/embeddings/dataset_2/network_based/node2vec_v1_32_dataset2_dataset.parquet


graphsage_v1_64_dataset2_dataset.parquet    shape=(1444, 67)  -> src/data/embeddings/dataset_2/feature_based/graphsage_v1_64_dataset2_dataset.parquet


node2vec_v1_64_dataset2_dataset.parquet     shape=(1444, 67)  -> src/data/embeddings/dataset_2/network_based/node2vec_v1_64_dataset2_dataset.parquet


graphsage_v1_128_dataset2_dataset.parquet   shape=(1444, 131)  -> src/data/embeddings/dataset_2/feature_based/graphsage_v1_128_dataset2_dataset.parquet


node2vec_v1_128_dataset2_dataset.parquet    shape=(1444, 131)  -> src/data/embeddings/dataset_2/network_based/node2vec_v1_128_dataset2_dataset.parquet


## Output summary

In [5]:
for f in sorted((EMB_ROOT/'dataset_2'/'feature_based').glob('graphsage_v1_*_dataset2_dataset.parquet')) + \
         sorted((EMB_ROOT/'dataset_2'/'network_based').glob('node2vec_v1_*_dataset2_dataset.parquet')):
    print(f'{f.parent.name}/{f.name:42s}  shape={pd.read_parquet(f).shape}')

feature_based/graphsage_v1_128_dataset2_dataset.parquet   shape=(1444, 131)
feature_based/graphsage_v1_32_dataset2_dataset.parquet    shape=(1444, 35)
feature_based/graphsage_v1_64_dataset2_dataset.parquet    shape=(1444, 67)
network_based/node2vec_v1_128_dataset2_dataset.parquet    shape=(1444, 131)
network_based/node2vec_v1_32_dataset2_dataset.parquet     shape=(1444, 35)
network_based/node2vec_v1_64_dataset2_dataset.parquet     shape=(1444, 67)
